# 02 - IEEE-CIS Predictive Model Benchmarks

In [ ]:
from pathlib import Path
import json
import os
import sys

def find_project_root() -> Path:
    candidates = [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        if (candidate / "src").is_dir() and (candidate / "configs").is_dir():
            return candidate
    for base in (Path("/kaggle/working"), Path("/kaggle/input")):
        if base.exists():
            matches = [path for path in base.glob("**/configs") if path.is_dir()]
            if matches:
                return matches[0].parent
    raise FileNotFoundError("Project root with src/ and configs/ was not found")

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

QUICK_RUN = os.getenv("THESIS_QUICK_RUN", "0") == "1"
ALLOW_SYNTHETIC_FALLBACK = os.getenv("THESIS_SYNTHETIC_FALLBACK", "0") == "1"
KAGGLE = Path("/kaggle").exists()
OUTPUT_BASE = Path("/kaggle/working/thesis_outputs") if KAGGLE else PROJECT_ROOT / "results/runs/notebooks"

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 120)
print({"project_root": str(PROJECT_ROOT), "quick_run": QUICK_RUN, "kaggle": KAGGLE})

## Thiết lập

- PR-AUC is the primary metric.
- F2 is the default operating-threshold objective.
- Full mode repeats the benchmark over three independent seeds.
- Quick mode is an explicit smoke test, not thesis evidence.

In [ ]:
from src.experiment import run_repeated_predictive_benchmarks

output_dir = OUTPUT_BASE / "02_ieee_cis_model_benchmarks"
result = run_repeated_predictive_benchmarks(
    PROJECT_ROOT / "configs/ieee_cis.yaml",
    output_dir=output_dir,
    model_names=("mlp", "tabular_resnet", "tree"),
    quick_run=QUICK_RUN,
    synthetic_fallback=ALLOW_SYNTHETIC_FALLBACK,
)
metrics = result["metrics"]
summary = result["summary"]
print({"data_sources": result["data_sources"], "seeds": result["seeds"], "output_dir": str(output_dir)})
display(summary.round(4))

## Data

In [ ]:
display(result["data_summary"])

## Results

In [ ]:
test_metrics = summary.query("split == 'test'").copy()
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(test_metrics["model"], test_metrics["pr_auc_mean"], yerr=test_metrics["pr_auc_std"].fillna(0), color="#4C72B0", capsize=4)
axes[0].set_title("Test PR-AUC mean ± SD")
axes[0].tick_params(axis="x", rotation=20)
axes[1].bar(test_metrics["model"], test_metrics["fbeta_mean"], yerr=test_metrics["fbeta_std"].fillna(0), color="#55A868", capsize=4)
axes[1].set_title("Test F2 mean ± SD")
axes[1].tick_params(axis="x", rotation=20)
plt.tight_layout()
fig.savefig(output_dir / "predictive_model_comparison.png", dpi=160, bbox_inches="tight")
plt.show()

In [ ]:
for seed, model_histories in result["histories"].items():
    for model_name, history in model_histories.items():
        display(Markdown(f"### {model_name} training history - seed {seed}"))
        display(pd.DataFrame(history))

## Takeaways

In [ ]:
best = test_metrics.sort_values("pr_auc_mean", ascending=False).iloc[0]
display(Markdown(
    f"- Data source: **{', '.join(result['data_sources'])}**.\n"
    f"- Seeds: **{result['seeds']}**.\n"
    f"- Highest mean test PR-AUC: **{best['model']} = {best['pr_auc_mean']:.4f} ± {best['pr_auc_std']:.4f}**.\n"
    f"- Mean locked-threshold F2: **{best['fbeta_mean']:.4f} ± {best['fbeta_std']:.4f}**.\n"
    "- Interpret only real-data, non-quick runs as thesis evidence."
))